# LSTM-arithmetic



In [ ]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

data_path = '/content/drive/MyDrive/Assignment2'

In [ ]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,(40*0)-18=,-18
1,6+4+31=,41
2,4*11+13=,57
3,44*39*26=,44616
4,(39*42)+27=,1665


In [ ]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [ ]:
char_to_id = {}
id_to_char = {}

# write your code here
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite
# GenAI assistance: ChatGPT helped me check which special tokens were required
# and I used a fixed 18-token vocabulary so the dictionary can run independently in leaderboard testing.
tokens = [
    '<pad>', '<eos>',
    '0', '1', '2', '3', '4',
    '5', '6', '7', '8', '9',
    '+', '-', '*', '(', ')', '='
]
char_to_id = {char: idx for idx, char in enumerate(tokens)}
id_to_char = {idx: char for char, idx in char_to_id.items()}
vocab_size = len(char_to_id)
print('Vocab size{}'.format(vocab_size))


Vocab size18


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [ ]:
# Write your code here
# GenAI assistance: ChatGPT helped me understand how the labels should align with
df_train['char_id_list'] = df_train['src'].apply(
    lambda x: [char_to_id[c] for c in x]
)

df_train['label_id_list'] = df_train.apply(
    lambda row:
        [char_to_id['<pad>']] * row['src'].index('=') +
        [char_to_id[c] for c in row['tgt']] +
        [char_to_id['<eos>']],
    axis=1
)
df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,(40*0)-18=-18,-18,13,"[15, 6, 2, 14, 2, 16, 13, 3, 10, 17, 13, 3, 10]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 13, 3, 10, 1]"
1,6+4+31=41,41,9,"[8, 12, 6, 12, 5, 3, 17, 6, 3]","[0, 0, 0, 0, 0, 0, 6, 3, 1]"
2,4*11+13=57,57,10,"[6, 14, 3, 3, 12, 3, 5, 17, 7, 9]","[0, 0, 0, 0, 0, 0, 0, 7, 9, 1]"
3,44*39*26=44616,44616,14,"[6, 6, 14, 5, 11, 14, 4, 8, 17, 6, 6, 8, 3, 8]","[0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 8, 3, 8, 1]"
4,(39*42)+27=1665,1665,15,"[15, 5, 11, 14, 6, 4, 16, 12, 4, 9, 17, 3, 8, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 8, 8, 7, 1]"


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [ ]:
batch_size = 64
epochs = 18
embed_dim = 256
hidden_dim = 256
lr = 0.001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # return the amount of data
        # GenAI assistance: ChatGPT helped me debug the Dataset indexing and the earlier
        return len(self.sequences)# Write your code here


    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        x = self.sequences.iloc[index]['char_id_list'] # Write your code here

        y = self.sequences.iloc[index]['label_id_list']# Write your code here

        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [ ]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [ ]:
# Build dataloader of train set and eval set, collate_fn is the collate function
# GenAI assistance: ChatGPT helped me verify the DataLoader syntax.
dl_train =torch.utils.data.DataLoader(
    ds_train,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
) # Write your code here


# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [ ]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        # Write your code here
        # Design your own model, which should contain embedding layer, several hidden layers and a linear layer
        # GenAI assistance: ChatGPT helped me compare RNN and LSTM for this sequence task.
        # I chose a 2-layer LSTM to better preserve information across the arithmetic expression
        self.embedding = torch.nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=char_to_id['<pad>']
        )

        self.rnn_layer1 = torch.nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.rnn_layer2 = torch.nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.linear = torch.nn.Sequential(
            torch.nn.Linear(
                in_features=hidden_dim,
                out_features=hidden_dim
            ),
            torch.nn.ReLU(),
            torch.nn.Linear(
                in_features=hidden_dim,
                out_features=vocab_size
            )
        )

    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)

    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)

        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)

        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)

        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)

        batch_x = self.linear(batch_x)

        return batch_x

    def generator(self, start_char, max_len=200):

        char_list = [char_to_id[c] for c in start_char]

        next_char = None

        while len(char_list) < max_len:
            # Write your code here
            # Pack the char_list to tensor
            # Input the tensor to the embedding layer, RNN layers, linear respectively
            # GenAI assistance: ChatGPT helped me debug the autoregressive generation loop
            x = torch.tensor(
             char_list,
             dtype=torch.long,
             device=next(self.parameters()).device
            ).unsqueeze(0)
            with torch.no_grad():
             y =self.embedding(x) # Obtain the next token prediction y
             y, _ = self.rnn_layer1(y)
             y, _ = self.rnn_layer2(y)
             y = self.linear(y)
            next_char =torch.argmax(
            y[0, -1, :],
            dim=-1
            ).item()# Use argmax function to get the next token prediction

            if next_char == char_to_id['<eos>']:
                break

            char_list.append(next_char)

        return [id_to_char[ch_id] for ch_id in char_list]

In [ ]:
torch.manual_seed(2)


device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)# Write your code here. Specify a device (cuda or cpu)

model = CharRNN(vocab_size,
         embed_dim,
         hidden_dim)

In [ ]:
criterion =  torch.nn.CrossEntropyLoss(
    ignore_index=char_to_id['<pad>']
)# Write your code here. Cross-entropy loss function. The loss function should ignore <pad>
optimizer =  torch.optim.Adam(
    model.parameters(),
    lr=lr
)# Write your code here. Use Adam or AdamW for Optimizer

# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [ ]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device)
model.train()
i = 0
for epoch in range(1, epochs+1):
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        # Write your code here
        # GenAI assistance: ChatGPT helped me verify the PyTorch training-step order.
        # Clear the gradient
        optimizer.zero_grad()
        batch_pred_y = model(batch_x.to(device), batch_x_lens)

        # Write your code here
        # Input the prediction and ground truths to loss function
        loss = criterion(
            batch_pred_y.reshape(-1, vocab_size),
            batch_y.to(device).reshape(-1)
        )
        # Back propagation
        loss.backward()


        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) # gradient clipping

        # Write your code here
        # Optimize parameters in the model
        optimizer.step()
        i+=1
        if i%50==0:
            bar.set_postfix(loss = loss.item())

    # Evaluate your model
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        batch_x = row['src']
        batch_y = row['tgt']

        prediction =  model.generator(batch_x)

        prediction = ''.join(
        prediction[len(batch_x):]
        )

        if prediction == batch_y:
         matched += 1

        total += 1

        bar_eval.set_postfix(
        accuracy=matched / total
        )# An example of using generator: model.generator('1+1=')

        # Write your code here. Input the batch_x to the model and generate the predictions
        # GenAI assistance: ChatGPT helped me check the Exact Match evaluation logic.
        # Write your code here.
        # Check whether the prediction match the ground truths
        # Compute exact match (EM) on the eval dataset
        # EM = correct/total


    print(matched/total)
    torch.save(
        model.state_dict(),
        "checkpoint.pt"
    )

Train epoch 1: 100%|██████████| 15625/15625 [03:11<00:00, 81.41it/s, loss=0.482]
Validation epoch 1: 10000it [00:54, 183.34it/s, accuracy=0.493]


0.4932


Train epoch 2: 100%|██████████| 15625/15625 [03:13<00:00, 80.67it/s, loss=0.357]
Validation epoch 2: 10000it [00:54, 183.48it/s, accuracy=0.566]


0.5655


Train epoch 3: 100%|██████████| 15625/15625 [03:14<00:00, 80.41it/s, loss=0.355]
Validation epoch 3: 10000it [00:54, 181.94it/s, accuracy=0.616]


0.6157


... (26 outputs left out)

Train epoch 17: 100%|██████████| 15625/15625 [03:15<00:00, 79.85it/s, loss=0.115]
Validation epoch 17: 10000it [00:56, 175.52it/s, accuracy=0.839]


0.8391


Train epoch 18: 100%|██████████| 15625/15625 [03:17<00:00, 79.24it/s, loss=0.0961]
Validation epoch 18: 10000it [00:55, 180.24it/s, accuracy=0.846]


0.8459


Running environment:Colab

Python version:Colab

##### 1. Describe your model architecture, data preprocessing methods, and hyperparameter settings, and justify your choices.  (5%)

Answer:

 I used a character-level 2-layer LSTM sequence generation model. The model consists of an embedding layer, two LSTM layers, and a fully connected output network. The embedding dimension is 256, and each LSTM layer has a hidden dimension of 256. After the LSTM layers, I used a Linear-ReLU-Linear structure to map the hidden representations to the 18-token vocabulary.

The vocabulary contains 18 tokens: digits 0-9, operators +, -, *, parentheses ( and ), the equal sign =, and the special tokens <pad> and <eos>.

For data preprocessing, I first converted the target answer into a string and appended it to the arithmetic expression for training. Each character was then converted into its corresponding token ID. In the target sequence, positions before the = symbol were replaced by <pad>, because the model only needs to learn predictions after =. The correct answer was followed by an <eos> token so that the model could learn when to stop generation.

The model also uses pack_padded_sequence and pad_packed_sequence to handle sequences with different lengths efficiently.

The hyperparameter settings used in my final experiment were:

Batch size: 64

Epochs: 18

Embedding dimension: 256

Hidden dimension: 256

Learning rate: 0.001

Gradient clipping value: 1

Optimizer: Adam

Loss function: CrossEntropyLoss, with <pad> ignored

I selected LSTM instead of a vanilla RNN because arithmetic expressions require the model to preserve information about earlier digits, operators, and parentheses. LSTM uses memory cells and gating mechanisms that can better preserve information over longer sequences and reduce the vanishing-gradient problem.

Teacher forcing was used during training. The ground-truth answer characters were included in the training input, allowing the model to learn the probability of each next token based on the correct previous sequence.

After 18 epochs, the model achieved an Exact Match accuracy of 84.59% on the 10,000-sample evaluation set.

##### 2. If you use 2-layer LSTM or GRU instead of RNN, what will happen to the quality of your answer generation? Why? (10%)

Answer:

I conducted an additional experiment to compare a **2-layer vanilla RNN** with the **2-layer LSTM**.

Both models used the same main hyperparameters and were trained for **5 epochs**.

| Model | Epochs | Exact Match Accuracy |
|---|---:|---:|
| 2-layer RNN | 5 | 15.72% |
| 2-layer LSTM | 5 | 66.84% |

The 2-layer LSTM achieved **66.84%**, while the vanilla RNN achieved only **15.72%**. Therefore, the LSTM improved the Exact Match accuracy by **51.12 percentage points**.

The main reason is that a vanilla RNN has difficulty preserving information over long sequences because of the **vanishing gradient problem**. Arithmetic expressions may contain multiple numbers, operators, and parentheses, so information from earlier tokens must still be remembered when generating the final answer.

LSTM uses memory cells and gating mechanisms, including the **input gate, forget gate, and output gate**, which help preserve useful information over longer sequences.

Therefore, both the experimental results and the model architecture show that the **2-layer LSTM performs much better than the 2-layer vanilla RNN** for this arithmetic sequence-generation task.

##### 3. If we construct an training set using three-digit numbers while the evaluation set is constructed from two-digit numbers, what will happen to the quality of your answer generation? (10%)

Answer:

I conducted an additional experiment in which the training data contained only **three-digit operands from 100 to 999**, while the model was evaluated using the original evaluation set.

The same **2-layer LSTM** architecture and the same main hyperparameters were used, and the model was trained for **5 epochs**.

| Training Setting | Exact Match Accuracy |
|---|---:|
| Original training data | 66.84% |
| Three-digit training data | 0.06% |

The Exact Match accuracy decreased from **66.84% to 0.06%**, which is a reduction of **66.78 percentage points**.

Some generated examples were:

- `34+47-32=` → Target: `49`, Prediction: `424`
- `33-41-15=` → Target: `-23`, Prediction: `-112`
- `31-(36*44)=` → Target: `-1553`, Prediction: `-155862`

This result shows a serious **distribution shift** between the training and evaluation data.

Although two-digit arithmetic problems may be mathematically easier than three-digit problems, the LSTM does not behave like a symbolic calculator. Instead, it learns character patterns, digit positions, sequence lengths, and answer distributions from the training data.

Because the model was trained only on three-digit operands, it became accustomed to longer numbers and generally larger output values. Therefore, it could not generalize well to the shorter numbers in the original evaluation set.

Overall, this experiment shows that training on numerically more difficult examples does **not necessarily guarantee better generalization** when the training and evaluation distributions are very different.

##### 4. If we construct a training set that includes 20% incorrect answers, how will this affect the quality of the generated responses? Present some examples. (10%)

Answer:

To study the effect of incorrect training labels, I intentionally changed **20% of the answers in the training set** to incorrect values.

The same **2-layer LSTM** architecture and the same main hyperparameters were used, and the model was trained for **5 epochs**. The model was then evaluated using the original clean evaluation set.

| Training Data | Exact Match Accuracy |
|---|---:|
| Clean training data | 66.84% |
| Training data with 20% incorrect answers | 60.40% |

The Exact Match accuracy decreased from **66.84% to 60.40%**, which is a reduction of **6.44 percentage points**.

Some incorrect generated examples were:

- `33-41-15=` → Target: `-23`, Prediction: `-25`
- `31-(36*44)=` → Target: `-1553`, Prediction: `-1541`
- `21*(41+3)=` → Target: `924`, Prediction: `936`
- `(46*38)+25=` → Target: `1773`, Prediction: `1767`
- `(7*33)+6=` → Target: `237`, Prediction: `235`

The incorrect answers introduce **label noise** into the training data. During teacher forcing, the model treats the provided target answers as ground truth. Therefore, when incorrect answers are included, the model also learns from these incorrect target sequences.

However, the accuracy did not decrease by the full 20%. This is because the remaining **80% of the training data was still correct**, so the model could still learn many useful arithmetic patterns.

Overall, the experiment shows that incorrect training labels reduce the reliability of the learned model and decrease the quality of the generated answers.

##### 5. Why do we need gradient clipping during training? (5%)

Answer:
Gradient clipping is used to reduce the exploding-gradient problem, which can occur when training recurrent neural networks such as RNNs and LSTMs.

During backpropagation through time, gradients are repeatedly propagated across many time steps. In some cases, repeated multiplication can cause the gradient values to become extremely large. If these large gradients are directly used to update the model parameters, a single optimization step may make a very large parameter change and cause unstable training.

In this experiment, I set the gradient clipping value to 1 and applied gradient clipping after loss.backward() and before optimizer.step().

Therefore, excessively large gradients are limited before the parameters are updated. This helps stabilize the optimization process and reduces the possibility of sudden unstable updates or numerical problems such as NaN loss.

Gradient clipping does not directly improve the arithmetic reasoning ability of the model, but it makes the training process of recurrent models more stable.

##### … Anything that can strengthen your report. (5%)

Answer:
The evaluation results show that the model continued to improve as training progressed. The Exact Match accuracy increased from 49.32% at epoch 1 to 84.59% at epoch 18, which is an improvement of 35.27 percentage points. This indicates that the 2-layer LSTM gradually learned useful arithmetic sequence patterns from the training data.

Although there was a small fluctuation around the middle of training, the overall trend remained upward, and the best performance was achieved at epoch 18. This suggests that the model had not clearly stopped improving within the current training range.

At the final epoch, the model correctly generated the complete answer for about 84.59% of the 10,000 evaluation samples. Since Exact Match is a strict evaluation metric, even a one-digit error causes the entire prediction to be counted as incorrect. Therefore, this result shows that the model was able to generate fully correct answers for a large proportion of the evaluation set.

However, approximately 15.41% of the evaluation samples were still incorrect. This suggests that the LSTM learned many useful arithmetic patterns, but it still does not behave like a perfectly reliable symbolic calculator. More training, different model architectures, or further hyperparameter tuning may improve the performance.

##### Generative AI Usage Statement

Answer:

I used ChatGPT as an assistant during this assignment. It was mainly used to help me understand the assignment requirements, clarify PyTorch concepts, check the logic of my implementation, and debug issues related to data preprocessing, Dataset indexing, sequence generation, checkpoint loading, and Exact Match evaluation.

ChatGPT was also used to help organize and improve the wording of the report and to design additional experiments for Questions 2, 3, and 4. I executed the experiments myself in Google Colab and used the actual experimental outputs in the final report.

The final model architecture, hyperparameter settings, training execution, experiment execution, and submission files were reviewed and selected by me.
